# Paramètre d’entrée

In [ ]:
# Parameters
# Parameters
input_path = "/app/data/clean/batch_118_clean.csv"  # Valeur par défaut # Papermill va écraser cette valeur
output_path = "/app/eda_reports/"  # optionnel

# Import des librairies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configurations graphiques
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12,6)

# Crée un dossier pour les plots si nécessaire
plots_dir = "/app/reports/eda_html/plots"
os.makedirs(plots_dir, exist_ok=True)

# Chargement du CSV

In [ ]:
# Lecture du CSV
df = pd.read_csv(input_path)
df.head()

# Statistiques descriptives

In [ ]:
# Statistiques globales
desc_stats = df.describe(include='all')
desc_stats.to_html(os.path.join(plots_dir, "stats.html"))
desc_stats

# Analyse des valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).sort_values(ascending=False)
missing_percent.to_frame("missing_percent").to_html(os.path.join(plots_dir, "missing.html"))

# Graphique des valeurs manquantes
plt.figure(figsize=(12,6))
sns.barplot(x=missing_percent.index, y=missing_percent.values)
plt.xticks(rotation=90)
plt.ylabel("Pourcentage manquant (%)")
plt.title("Valeurs manquantes par colonne")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "missing_plot.png"))
plt.close()

# Distribution des colonnes numériques

In [ ]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns

for col in numeric_cols:
    plt.figure()
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f"Distribution: {col}")
    plt.savefig(os.path.join(plots_dir, f"{col}_dist.png"))
    plt.close()

# Statistiques par catégorie

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    counts = df[col].value_counts()
    counts.plot(kind='bar')
    plt.title(f"Répartition: {col}")
    plt.ylabel("Nombre")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{col}_bar.png"))
    plt.close()

# Corrélation entre variables numériques

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "correlation_matrix.png"))
plt.close()

# Export HTML

In [ ]:
summary_html_path = os.path.join(plots_dir, "summary.html")
with open(summary_html_path, "w") as f:
    f.write("<h1>Résumé EDA</h1>")
    f.write("<h2>Statistiques descriptives</h2>")
    f.write(desc_stats.to_html())
    f.write("<h2>Valeurs manquantes (%)</h2>")
    f.write(missing_percent.to_frame("missing_percent").to_html())
    f.write("<h2>Corrélations</h2>")
    f.write('<img src="correlation_matrix.png">')
    f.write("<h2>Colonnes catégorielles</h2>")
    for col in categorical_cols:
        f.write(f"<h3>{col}</h3><img src='{col}_bar.png'>")